# Rivet on the Radeon PRO W7900

**Verified multimodal ad creation on one AMD Radeon GPU.**

A product image, brand kit and brief become a verified three-scene vertical advertisement. The
product, logo and text never pass through a generative model, and every export carries a Campaign
Receipt: input hashes, seeds, per-stage timings, peak VRAM and ten audit checks.

This notebook reproduces every number in the submission. Run the cells in order.

| Cell | What it does | Roughly |
|---|---|---|
| 1 | Report the GPU and ROCm stack | seconds |
| 2 | Install Rivet without disturbing the ROCm torch | 2-4 min |
| 3 | Fetch models at the revisions in `MODEL_LICENSES.md` | 20-30 min first time, seconds after |
| 4 | Produce all evidence: tests, gates, benchmarks | 40-60 min |
| 5 | Show the receipt and the advertisement | seconds |

Set `MODEL_DIR` in cell 1 to a persistent path so a later session skips the download.

In [ ]:
import os, subprocess, pathlib

# Persist the model cache so a second session does not pay the download again.
MODEL_DIR = os.environ.get("MODEL_DIR", "/workspace/models")
pathlib.Path(MODEL_DIR).mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = MODEL_DIR
os.environ["HF_HUB_CACHE"] = f"{MODEL_DIR}/hub"
print("model cache:", os.environ["HF_HUB_CACHE"])

print(subprocess.run(["bash", "-lc", "rocm-smi --showproductname 2>/dev/null | head -20 || true"],
                     capture_output=True, text=True).stdout)
try:
    import torch
    print("torch    :", torch.__version__)
    print("hip      :", getattr(torch.version, "hip", None) or "n/a")
    print("available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        p = torch.cuda.get_device_properties(0)
        print("device   :", torch.cuda.get_device_name(0))
        print("vram     :", p.total_memory // (1024**2), "MB")
except ImportError:
    print("torch not importable from the notebook kernel")

## 2. Install

The instance ships a ROCm build of PyTorch. Installing a torch from PyPI would replace it with the
CPU or CUDA build and the GPU would silently disappear, so the bootstrap installs into a virtual
environment that can see the system packages and **aborts if torch changed**.

In [ ]:
!./scripts/cloud/bootstrap.sh 2>&1 | tail -40

## 3. Models

Pinned to the exact revisions recorded in `MODEL_LICENSES.md`, so the weights used are provably the
weights licensed. SDXL ships fp32, fp16 and single-file copies of the same network; only the fp16
variant Rivet loads is downloaded.

In [ ]:
!.venv/bin/python scripts/models/fetch.py

## 4. Evidence

Runs the whole suite: environment, tests, golden determinism, the five submission gates, the offline
gate with every outbound socket blocked, and the cold, hot and residency benchmarks. Each stage is
committed as it finishes, so an interrupted session keeps what it already paid for.

Results land in `docs/evidence/` and `docs/benchmarks/`.

In [ ]:
!source .venv/bin/activate && ./scripts/cloud/evidence.sh 2>&1 | tail -50

## 5. The result

The receipt is the product surface: every check, what it observed, and the hash over the whole record.

In [ ]:
import json, pathlib
from IPython.display import Video, Image, display

receipts = sorted(pathlib.Path(".evidence").rglob("receipt.json"))
if not receipts:
    print("no receipt yet — run cell 4")
else:
    receipt = json.loads(receipts[-1].read_text())
    print(f"receipt {receipt['receipt_hash'][:16]}  passed={receipt['passed']}")
    for scene in receipt["scenes"]:
        for check in scene["checks"]:
            mark = "ok  " if check["passed"] else "FAIL"
            flag = " (advisory)" if check.get("advisory") else ""
            print(f"  [{mark}] {scene['shot_id']:<6} {check['check_id']} "
                  f"{check['metric']}: {check['observed']}{flag}")
    for scene in receipt["scenes"]:
        still = pathlib.Path(scene["still_path"])
        if still.is_file():
            display(Image(filename=str(still), width=260))
    video = receipt.get("video_path")
    if video and pathlib.Path(video).is_file():
        display(Video(video, width=320, embed=True))